# CADMUS — Quickstart Notebook

Circuit-Aware Dynamic Multilayer Update System

Este notebook muestra el uso completo de CADMUS en AerSimulator.
Para usar con IBM Quantum real, sustituye `backend='aer_simulator'` por tu backend de ibm_torino.

In [ ]:
# Instalar si es necesario
# !pip install cadmus-qec

In [ ]:
from qiskit import QuantumCircuit
from cadmus import CADMUS, CircuitHealthCheck, NoiseProfiler

# Circuito de prueba: GHZ de 3 qubits
qc = QuantumCircuit(3, 3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.measure([0, 1, 2], [0, 1, 2])

qc.draw('mpl')

## 1. Circuit Health Check

In [ ]:
health = CircuitHealthCheck(backend='aer_simulator')
report = health.score(qc)

print(f"Viability score : {report['viability']:.3f}")
print(f"Depth penalty   : {report['depth_penalty']:.3f}")
print(f"Gate error      : {report['gate_error_budget']:.3f}")
print(f"TVD estimate    : {report['tvd_estimate']:.3f}")
print(f"Recommendation  : {report['recommendation']}")

## 2. Noise Profiling

In [ ]:
from qiskit_aer import AerSimulator

backend = AerSimulator()
profiler = NoiseProfiler(backend)
noise_map = profiler.profile(qc)

print(profiler.summary(noise_map))

## 3. Full CADMUS Pipeline

In [ ]:
cadmus = CADMUS(
    backend='aer_simulator',
    layers=['A', 'B', 'C'],
    drift_window=50,
    drift_threshold=0.15,
    verbose=True
)

result = cadmus.run(qc, shots=1024)
print(result.summary())

In [ ]:
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_histogram(result.raw_counts, ax=axes[0], title='Raw (NISQ)')
plot_histogram(result.counts, ax=axes[1], title='CADMUS Corrected')

plt.tight_layout()
plt.show()

## 4. Syndrome Drift Detection

In [ ]:
from cadmus import SyndromeDriftDetector

detector = SyndromeDriftDetector(window=20, threshold=0.1)

# Simular sindromes estables
for _ in range(30):
    detector.update([0, 0, 0, 0])

# Simular drift
for _ in range(30):
    detector.update([1, 1, 0, 1])

print(detector.summary())
print(f'Drift detected: {detector.drift_detected()}')